# Financial Fraud Detector using GraphSAGE


Graph Structure (Node Classification for Account Takeover Fraud)

Model: GNN - Graph Neural Network

Architecture: GraphSAGE while using GraphSAINT as the sampler

Nodes: nameOrig and nameDest (Accounts)

Edges: transaction between the accounts - meaning the transaction between nameOrig and nameDest (nameOrig -> nameDest)

Edge features: step, type, amount, oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest.

Nodes labels: isFraud and isFlaggedFraud

Graph type: Homogeneous graph

Epochs: 50

Nodes feature: 6 feature (2 degree features, 3 amount‑pattern features(transaction behavior patterns), unique counterparties)


## HuggingFace Login

In [1]:
from huggingface_hub import login

#you get this from creating an account on huggingface.co and generating a token in your account settings.
login(token="")


## Imports

In [2]:
!pip install torch_geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from datasets import load_dataset
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.utils import degree
from torch_geometric.loader import GraphSAINTRandomWalkSampler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# ---- GPU Memory Cleanup ----
torch.cuda.empty_cache()

## Load Dataset

In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'],
        num_rows: 21000000
    })
})


## Data Inspection

In [4]:
import pandas as pd

column_names = [
    'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
    'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest',
    'isFraud', 'isFlaggedFraud'
]

df = ds["train"].to_pandas()[column_names]
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,371,CASH_OUT,367336.05,sdv-pii-r8zd6,4514816.83,2108392.86,sdv-pii-q6998,1265486.06,2454140.46,0,0
1,368,TRANSFER,238.63,sdv-pii-xq6z3,430944.71,1865444.60,sdv-pii-n2ql8,107927.46,2021.16,0,0
2,141,CASH_OUT,254.93,sdv-pii-805w0,839593.53,8008353.88,sdv-pii-yo0z6,773352.22,20.79,0,0
3,191,CASH_IN,501547.39,sdv-pii-279tw,41226.40,28633.52,sdv-pii-9zlyl,6825363.55,16442078.24,0,0
4,169,TRANSFER,71832.00,sdv-pii-ksz58,248694.60,793617.86,sdv-pii-0ykbo,579313.76,829850.96,0,0
...,...,...,...,...,...,...,...,...,...,...,...
20999995,198,CASH_OUT,13315.94,C639852412,2423268.89,12413862.98,M1385592393,1504614.61,0.10,0,0
20999996,198,CASH_IN,44194.38,C1479656058,815049.21,4148521.18,M2086295095,183371.08,0.00,0,0
20999997,122,PAYMENT,79701.00,C1817049050,0.08,11665.50,C1033381601,769833.04,0.08,0,0
20999998,139,TRANSFER,473635.45,C532984095,1759699.35,153806.01,C1680845012,641670.03,108828.76,0,0


## Preprocessing

In [5]:
#pre-processing
df = df.drop_duplicates()

print(df.isna().sum())

#Fill NA only in numeric columns if needed
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

#Check constant columns
constant_columns = [col for col in df.columns if df[col].nunique() == 1]
print("Constant columns found:", constant_columns)

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
Constant columns found: []


## Encode Categorical Column

In [6]:
df["type"] = df["type"].astype("category").cat.codes
df["nameOrig"] = df["nameOrig"].astype("category")
df["nameDest"] = df["nameDest"].astype("category")

df["src_id"] = df["nameOrig"].cat.codes
df["dst_id"] = df["nameDest"].cat.codes

num_nodes = max(df["src_id"].max(), df["dst_id"].max()) + 1

## Extract Edge Index

In [7]:
#extract edge index
edge_index = torch.tensor(df[["src_id", "dst_id"]].values.T, dtype=torch.long)
print(edge_index[:, :10])

tensor([[10076193, 11103076,  7023000,  6102980,  9052561,  9363349,  8239305,
          9260416,  7275688,  8990751],
        [ 6659859,  6168325,  8007478,  4091620,  2659415,  2781906,  7936760,
          3776581,  7344028,  4356589]])


## Pytorch Graph (Full Graph)

In [8]:
edge_index = torch.tensor(df[["src_id", "dst_id"]].values.T, dtype=torch.long)

edge_attr = torch.tensor(df[[
    "step", "type", "amount",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest"
]].values, dtype=torch.float32)

data = Data(
    edge_index=edge_index,
    edge_attr=edge_attr,
    num_nodes=num_nodes
)

## Behavioral Node Features

In [9]:
deg_out = degree(edge_index[0], num_nodes=num_nodes)
deg_in = degree(edge_index[1], num_nodes=num_nodes)

df["balance_change"] = df["oldbalanceOrg"] - df["newbalanceOrig"]

tx_freq = df.groupby("src_id").size().reindex(range(num_nodes), fill_value=0)
avg_amount = df.groupby("src_id")["amount"].mean().reindex(range(num_nodes), fill_value=0)
avg_balance_change = df.groupby("src_id")["balance_change"].mean().reindex(range(num_nodes), fill_value=0)
unique_dest = df.groupby("src_id")["dst_id"].nunique().reindex(range(num_nodes), fill_value=0)

data.x = torch.stack([
    torch.log1p(deg_out),
    torch.log1p(deg_in),
    torch.log1p(torch.tensor(tx_freq.values)),
    torch.log1p(torch.tensor(avg_amount.values) - avg_amount.min() + 1e-6),
    torch.log1p(torch.tensor(avg_balance_change.values) - avg_balance_change.min() + 1e-6), # Corrected from log11p
    torch.log1p(torch.tensor(unique_dest.values))
], dim=1)

data.x = data.x.float()
data.edge_attr = (data.edge_attr - data.edge_attr.mean(dim=0)) / (data.edge_attr.std(dim=0) + 1e-6)

## Build Node Labels

In [10]:
node_labels = torch.zeros(num_nodes, dtype=torch.long)

fraud_edges = df["isFraud"] == 1
src_fraud = df["src_id"][fraud_edges].values
dst_fraud = df["dst_id"][fraud_edges].values

node_labels[src_fraud] = 1
node_labels[dst_fraud] = 1

data.y = node_labels

## Train/Val/Test Split (Node-level)

- 70% → training
- 15% → validation
- 15% → testing

In [11]:
perm = torch.randperm(num_nodes)
train_end = int(0.7 * num_nodes)
val_end = int(0.85 * num_nodes)

data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

data.train_mask[perm[:train_end]] = True
data.val_mask[perm[train_end:val_end]] = True
data.test_mask[perm[val_end:]] = True

## Handle Class Imbalance

In [12]:
import torch

class_counts = torch.bincount(data.y)
print(f"Class distribution: {class_counts.tolist()}")
print(f"Fraud ratio: {class_counts[1].item() / class_counts.sum().item():.4%}")

# Keep class weights as fallback reference
class_weights = 1.0 / (class_counts.float() + 1e-6)
class_weights = class_weights * (2 / class_weights.sum())


Class distribution: [11409624, 54476]
Fraud ratio: 0.4752%


In [13]:
import torch

# Get the version of torch and cuda to match the binaries
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'

# Reinstall the dependencies from the specialized provider
!pip install --no-index torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric
!pip install torch-scatter

!pip install SparseTensor
!pip install torch-sparse

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
ERROR: Could not find a version that satisfies the requirement torch-cluster (from versions: none)
ERROR: No matching distribution found for torch-cluster
ERROR: Could not find a version that satisfies the requirement SparseTensor (from versions: none)
ERROR: No matching distribution found for SparseTensor


In [14]:
import torch

# This ensures we get the exact string needed for the URL (e.g., 'cu121' or 'cpu')
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'

# We use --force-reinstall to ensure no old, broken binaries remain
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
-f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
  Using cached torch_cluster-1.6.3.tar.gz (54 kB)
  Installing build dependencies ... - \ | done
  Getting requirements to build wheel ... - error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [17 lines of output]
      Traceback (most recent call last):
        File "/home/if3/miniconda3/envs/G6/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/home/if3/miniconda3/envs/G6/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
        File "/home/if3/miniconda3/envs/G6/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_setting

In [15]:

from torch_geometric.loader import GraphSAINTNodeSampler
from torch_geometric.loader import GraphSAINTRandomWalkSampler

# Ensure 'data' is a torch_geometric.data.Data object
train_loader = GraphSAINTRandomWalkSampler(
    data,
    batch_size=8000,
    walk_length=4,
    num_steps=50,
    sample_coverage=5,
    save_dir="./graphsaint",
)


/home/if3/miniconda3/envs/G6/lib/python3.10/site-packages/torch_sparse/storage.py:155: RuntimeWarning: overflow encountered in scalar multiply
  max_value = self._sparse_sizes[0] * self._sparse_sizes[1]


## NeighborLoader for Mini-batch Training

## GraphSAGE Model

In [16]:
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x)


## Focal Loss

Focal Loss addresses class imbalance by down-weighting easy (non-fraud) examples
and focusing training on hard (fraud) examples.

**Formula:** `FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)`

- **γ (gamma)**: Focusing parameter. Higher γ → more focus on hard misclassified examples.
  - `γ=0` → standard cross-entropy
  - `γ=2` → recommended default for fraud detection
- **α (alpha)**: Class balancing weight (same as class_weights).


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss for binary/multi-class node classification.
    Focuses training on hard-to-classify (fraud) examples.

    Args:
        alpha (Tensor): Per-class weights (same role as class_weights in CrossEntropy).
        gamma (float): Focusing parameter. gamma=2 is recommended for fraud detection.
        reduction (str): 'none' | 'mean' | 'sum'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha        # shape: [num_classes]
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # Step 1: Standard cross-entropy (per sample, no reduction)
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')

        # Step 2: Compute p_t — probability of the TRUE class
        probs = F.softmax(logits, dim=1)                         # [N, C]
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)   # [N]

        # Step 3: Focal modulation factor: (1 - p_t)^gamma
        # Easy examples (p_t near 1) → factor near 0 → loss suppressed
        # Hard examples (p_t near 0) → factor near 1 → loss preserved
        focal_factor = (1.0 - p_t) ** self.gamma

        # Step 4: Apply focal factor
        focal_loss = focal_factor * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss  # 'none'


# ---- Instantiate Focal Loss ----
# alpha=class_weights: still penalises the majority class (non-fraud) less
# gamma=2: standard focusing strength; increase to 3-5 if recall is still low
FOCAL_GAMMA = 2.0
print(f"Focal Loss initialised with gamma={FOCAL_GAMMA}")
print("(class_weights will be passed as alpha when criterion is created in Training Setup)")


Focal Loss initialised with gamma=2.0
(class_weights will be passed as alpha when criterion is created in Training Setup)


## Training Setup

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = class_weights.to(device)

model = GraphSAGE(in_channels=6, hidden_channels=128, out_channels=2).to(device)

# ---- Focal Loss (replaces CrossEntropyLoss) ----
# alpha=class_weights balances majority/minority classes
# gamma=FOCAL_GAMMA focuses on hard fraud examples
criterion = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA, reduction='none')

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f"Using FocalLoss with gamma={FOCAL_GAMMA} on device={device}")


Using FocalLoss with gamma=2.0 on device=cuda


In [19]:
from torch_geometric.loader import NeighborLoader

eval_loader = NeighborLoader(
    data,
    num_neighbors=[-1],   # full neighborhood
    batch_size=50000,     # large but safe
    shuffle=False
)


/home/if3/miniconda3/envs/G6/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


## Evaluation Function

In [20]:
def evaluate(mask):
    model.eval()
    preds = torch.zeros(data.num_nodes, dtype=torch.long, device=device)

    with torch.no_grad():
        for batch in eval_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            preds[batch.n_id] = out.argmax(dim=1)

    y_true = data.y[mask].cpu()
    y_pred = preds[mask].cpu()

    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, zero_division=0),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0)
    )


## Training Loop

In [21]:
for epoch in range(1, 51):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)
        batch.x = batch.x.float()
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index)

        # GraphSAINT normalization combined with Focal Loss
        # criterion returns per-sample loss (reduction='none') so we can
        # apply GraphSAINT node_norm weights before summing
        valid = batch.node_norm > 0
        focal = criterion(out[valid], batch.y[valid])   # [N_valid]
        loss = (focal * batch.node_norm[valid]).sum()    # scalar

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_acc, train_f1, train_prec, train_rec = evaluate(data.train_mask)
    val_acc, val_f1, val_prec, val_rec = evaluate(data.val_mask)

    print(f"Epoch: {epoch:02d} | Loss: {total_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} F1: {train_f1:.4f} "
          f"Prec: {train_prec:.4f} Rec: {train_rec:.4f} | "
          f"Val Acc: {val_acc:.4f} F1: {val_f1:.4f} "
          f"Prec: {val_prec:.4f} Rec: {val_rec:.4f}")


Epoch: 01 | Loss: 0.8399 | Train Acc: 0.8986 F1: 0.0259 Prec: 0.0136 Rec: 0.2852 | Val Acc: 0.8982 F1: 0.0262 Prec: 0.0137 Rec: 0.2839
Epoch: 02 | Loss: 0.4979 | Train Acc: 0.5203 F1: 0.0116 Prec: 0.0059 Rec: 0.5951 | Val Acc: 0.5206 F1: 0.0118 Prec: 0.0060 Rec: 0.5957
Epoch: 03 | Loss: 0.4512 | Train Acc: 0.6222 F1: 0.0158 Prec: 0.0080 Rec: 0.6391 | Val Acc: 0.6218 F1: 0.0161 Prec: 0.0082 Rec: 0.6425
Epoch: 04 | Loss: 0.4600 | Train Acc: 0.3997 F1: 0.0125 Prec: 0.0063 Rec: 0.7993 | Val Acc: 0.4003 F1: 0.0127 Prec: 0.0064 Rec: 0.7973
Epoch: 05 | Loss: 0.4220 | Train Acc: 0.4111 F1: 0.0131 Prec: 0.0066 Rec: 0.8264 | Val Acc: 0.4113 F1: 0.0134 Prec: 0.0067 Rec: 0.8266
Epoch: 06 | Loss: 0.4326 | Train Acc: 0.7493 F1: 0.0205 Prec: 0.0104 Rec: 0.5533 | Val Acc: 0.7493 F1: 0.0209 Prec: 0.0107 Rec: 0.5557
Epoch: 07 | Loss: 0.4256 | Train Acc: 0.8444 F1: 0.0263 Prec: 0.0135 Rec: 0.4428 | Val Acc: 0.8442 F1: 0.0265 Prec: 0.0137 Rec: 0.4403
Epoch: 08 | Loss: 0.4372 | Train Acc: 0.7412 F1: 0.0202

## Table for Results

In [22]:
history = {
    "epoch": [],
    "train_acc": [],
    "train_f1": [],
    "train_precision": [],
    "train_recall": [],
    "val_acc": [],
    "val_f1": [],
    "val_precision": [],
    "val_recall": []
}


## Final Test Performance

In [23]:
test_acc, test_f1, test_prec, test_rec = evaluate(data.test_mask)
print(f"\nTest Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}, "
      f"Test Precision: {test_prec:.4f}, Test Recall: {test_rec:.4f}")



Test Acc: 0.8100, Test F1: 0.0248, Test Precision: 0.0127, Test Recall: 0.5097
